In [10]:
import pandas as pd 
data = pd.read_csv('../data/train.csv')

In [11]:
# ============================================
# REPRODUCTION DES MODIFICATIONS DE L'EDA
# ============================================

# Suppression de l'identifiant
data.drop("Id", axis=1, inplace=True)

# Variables pour lesquelles NaN signifie "absence"
cols_none = [
    "PoolQC",
    "MiscFeature",
    "Fence",
    "FireplaceQu",
    "Alley",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType"
]

# Remplacement des NaN par "None"
data[cols_none] = data[cols_none].fillna("None")

In [12]:
def create_all_features(df):
    """Crée les features les plus utiles (version simplifiée)"""
    df_feat = df.copy()
    
    
    # 1. QualityArea (découverte ML Mastery)
    df_feat['QualityArea'] = df_feat['OverallQual'] * df_feat['GrLivArea']
    
    # 2. TotalSF (surface totale habitable)
    df_feat['TotalSF'] = df_feat['TotalBsmtSF'] + df_feat['1stFlrSF'] + df_feat['2ndFlrSF']
    
    # 3. HouseAge (âge de la maison)
    df_feat['HouseAge'] = df_feat['YrSold'] - df_feat['YearBuilt']
    
    # 4. RemodAge (âge depuis la rénovation)
    df_feat['RemodAge'] = df_feat['YrSold'] - df_feat['YearRemodAdd']
    
    # 5. TotalBath (salles de bain totales)
    df_feat['TotalBath'] = (df_feat['FullBath'] + 0.5*df_feat['HalfBath'] +
                            df_feat['BsmtFullBath'] + 0.5*df_feat['BsmtHalfBath'])
    
    # 6. TotalPorchSF (surface des porches)
    df_feat['TotalPorchSF'] = (df_feat['OpenPorchSF'] + df_feat['EnclosedPorch'] + 
                                df_feat['3SsnPorch'] + df_feat['ScreenPorch'])
    
   
    return df_feat

print(" Fonction create_all_features() définie (version simplifiée)")

 Fonction create_all_features() définie (version simplifiée)


In [13]:
data_feat = create_all_features(data)

In [14]:
# ============================================
# SPLIT TRAIN/TEST (SUR LE TRAIN KAGGLE)
# ============================================

from sklearn.model_selection import train_test_split

# 1. Séparer X (features) et y (cible)
X = data_feat.drop('SalePrice', axis=1)
y = data_feat['SalePrice']

# 2. Split 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,          # 20% pour le test
    random_state=42
)

print(f"Train: {X_train.shape[0]} lignes")
print(f"Test: {X_test.shape[0]} lignes")

Train: 1168 lignes
Test: 292 lignes


In [15]:
# ============================================
# PREPROCESSING COMPLET + FEATURE SELECTION
# (Avec toutes les colonnes)
# ============================================

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Lasso
import numpy as np
import pandas as pd

print("="*60)
print("PREPROCESSING + FEATURE SELECTION")
print("="*60)

# ============================================
# 1. IMPUTATION DE TOUTES LES VALEURS MANQUANTES
# ============================================

print("\n Gestion des valeurs manquantes...")

# 1.1 LotFrontage - par quartier
if 'Neighborhood' in X_train.columns:
    X_train['LotFrontage'] = X_train.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )
    # Appliquer la même imputation sur le test en utilisant les médianes par quartier du train
    neighborhood_medians = X_train.groupby('Neighborhood')['LotFrontage'].median()
    for neighborhood in X_test['Neighborhood'].unique():
        if neighborhood in neighborhood_medians.index:
            mask = X_test['Neighborhood'] == neighborhood
            X_test.loc[mask, 'LotFrontage'] = X_test.loc[mask, 'LotFrontage'].fillna(
                neighborhood_medians[neighborhood]
            )
else:
    # Fallback : médiane globale
    median_lot = X_train['LotFrontage'].median()
    X_train['LotFrontage'] = X_train['LotFrontage'].fillna(median_lot)
    X_test['LotFrontage'] = X_test['LotFrontage'].fillna(median_lot)

print(" LotFrontage imputé")

# 1.2 MasVnrArea - si pas de parement, 0
if 'MasVnrType' in X_train.columns:
    X_train.loc[X_train['MasVnrType'].isna(), 'MasVnrArea'] = 0
    X_test.loc[X_test['MasVnrType'].isna(), 'MasVnrArea'] = 0
    
# Les valeurs restantes (si il y en a) par médiane
median_mas = X_train['MasVnrArea'].median()
X_train['MasVnrArea'] = X_train['MasVnrArea'].fillna(median_mas)
X_test['MasVnrArea'] = X_test['MasVnrArea'].fillna(median_mas)
print(" MasVnrArea imputé")

# 1.3 GarageYrBlt - si pas de garage, utiliser YearBuilt
if 'GarageType' in X_train.columns:
    # Si pas de garage (GarageType manquant), mettre 0
    X_train.loc[X_train['GarageType'].isna(), 'GarageYrBlt'] = 0
    X_test.loc[X_test['GarageType'].isna(), 'GarageYrBlt'] = 0
    
# Pour les valeurs restantes, utiliser YearBuilt
mask_train = X_train['GarageYrBlt'].isna()
X_train.loc[mask_train, 'GarageYrBlt'] = X_train.loc[mask_train, 'YearBuilt']

mask_test = X_test['GarageYrBlt'].isna()
X_test.loc[mask_test, 'GarageYrBlt'] = X_test.loc[mask_test, 'YearBuilt']
print(" GarageYrBlt imputé")

# 1.4 Electrical - par le mode (comme avant)
mode_electrical = X_train['Electrical'].mode()[0]
X_train['Electrical'] = X_train['Electrical'].fillna(mode_electrical).astype(str)
X_test['Electrical'] = X_test['Electrical'].fillna(mode_electrical).astype(str)
print(f" Electrical imputé par le mode : '{mode_electrical}'")

# ============================================
# VÉRIFICATION FINALE
# ============================================

print(f"\n Vérification des NaN restants :")
print(f"   Train : {X_train.isnull().sum().sum()} NaN")
print(f"   Test : {X_test.isnull().sum().sum()} NaN")

# Si encore des NaN, imputer par 0 pour les numériques et mode pour les catégories
if X_train.isnull().sum().sum() > 0:
    print("\n  Il reste des NaN, imputation d'urgence...")
    for col in X_train.columns:
        if X_train[col].dtype in ['int64', 'float64']:
            median_val = X_train[col].median()
            X_train[col] = X_train[col].fillna(median_val)
            X_test[col] = X_test[col].fillna(median_val)
        else:
            mode_val = X_train[col].mode()[0]
            X_train[col] = X_train[col].fillna(mode_val)
            X_test[col] = X_test[col].fillna(mode_val)

# ============================================
# 2. ENCODAGE DE TOUTES LES CATÉGORIES
# ============================================

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"\n Colonnes catégorielles : {len(cat_cols)}")

for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    le.fit(X_train[col])
    X_train[col] = le.transform(X_train[col])
    
    # Gérer les valeurs inconnues dans le test
    first_class = le.classes_[0]
    X_test[col] = X_test[col].apply(lambda x: x if x in le.classes_ else first_class)
    X_test[col] = le.transform(X_test[col])

print(f" {len(cat_cols)} colonnes encodées")

# ============================================
# 3. RÉCUPÉRER LES COLONNES NUMÉRIQUES (TOUTES)
# ============================================

# TOUTES les colonnes numériques (incluant les nouvelles features)
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'Id' in num_cols:
    num_cols.remove('Id')
if 'SalePrice' in num_cols:
    num_cols.remove('SalePrice')

print(f"\n Colonnes numériques : {len(num_cols)}")
print(f"   {num_cols[:10]}...")

# ============================================
# 4. STANDARDISATION DES NUMÉRIQUES
# ============================================

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

print(f" Colonnes numériques standardisées")

# ============================================
# 5. RÉASSEMBLER (NUMÉRIQUES + CATÉGORIES ENCODÉES)
# ============================================

# Récupérer les colonnes catégorielles encodées (en tant que tableau)
cat_encoded = X_train[cat_cols].values
X_train_cat = cat_encoded  # déjà encodées
X_test_cat = X_test[cat_cols].values

X_train_processed = np.hstack([X_train_num, X_train_cat])
X_test_processed = np.hstack([X_test_num, X_test_cat])

print(f"\n Train après preprocessing : {X_train_processed.shape}")
print(f" Test après preprocessing : {X_test_processed.shape}")


PREPROCESSING + FEATURE SELECTION

 Gestion des valeurs manquantes...
 LotFrontage imputé
 MasVnrArea imputé
 GarageYrBlt imputé
 Electrical imputé par le mode : 'SBrkr'

 Vérification des NaN restants :
   Train : 0 NaN
   Test : 0 NaN

 Colonnes catégorielles : 43
 43 colonnes encodées

 Colonnes numériques : 42
   ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2']...
 Colonnes numériques standardisées

 Train après preprocessing : (1168, 85)
 Test après preprocessing : (292, 85)


In [16]:

# ============================================
# 6. LASSO POUR LA SÉLECTION DE FEATURES
# ============================================

print("\n" + "="*60)
print("LASSO FEATURE SELECTION")
print("="*60)

# Option 1 : Lasso avec validation croisée
from sklearn.linear_model import LassoCV
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_processed, y_train)

# Option 2 : Lasso simple avec alpha choisi manuellement
# lasso = Lasso(alpha=0.001, random_state=42, max_iter=10000)
# lasso.fit(X_train_processed, y_train)

# Récupérer les coefficients
feature_names = num_cols + cat_cols
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso.coef_
})

# Filtrer les features importantes
important_features = coef_df[coef_df['Coefficient'] != 0].sort_values('Coefficient', key=abs, ascending=False)
print(f"\n {len(important_features)} features sélectionnées sur {len(feature_names)}")

print(f"\n Top 20 features les plus importantes :")
print(important_features.head(20))

# Sélectionner les colonnes à garder
selected_features = important_features['Feature'].tolist()
print(f"\n Features à garder : {len(selected_features)}")

# ============================================
# 7. CRÉER LES DATASETS FINAUX AVEC SEULEMENT LES FEATURES SÉLECTIONNÉES
# ============================================

# Garder uniquement les colonnes sélectionnées dans les DataFrames originaux
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"\n X_train_final : {X_train_selected.shape}")
print(f" X_test_final : {X_test_selected.shape}")

# Optionnel : Sauvegarder
# X_train_selected.to_csv('X_train_selected.csv', index=False)
# X_test_selected.to_csv('X_test_selected.csv', index=False)


LASSO FEATURE SELECTION

 40 features sélectionnées sur 85

 Top 20 features les plus importantes :
         Feature   Coefficient
36   QualityArea  26958.754659
3    OverallQual   8912.885028
25    GarageCars   8606.054295
72   KitchenQual  -8204.059313
63      BsmtQual  -5446.766720
0     MSSubClass  -5292.289799
40     TotalBath   4808.043851
60     ExterQual  -4027.674356
23    Fireplaces   3870.135507
65  BsmtExposure  -3279.865587
2        LotArea   3227.494582
38      HouseAge  -2702.760507
4    OverallCond   2311.863480
37       TotalSF   2117.760288
7     MasVnrArea   2064.868124
22  TotRmsAbvGrd   1901.682965
39      RemodAge  -1766.854995
31   ScreenPorch   1758.971430
73    Functional   1748.085716
27    WoodDeckSF   1641.711655

 Features à garder : 40

 X_train_final : (1168, 40)
 X_test_final : (292, 40)


In [ ]:
# ============================================
# CRÉATION DES DEUX VERSIONS DES DATASETS
# ============================================

print("\n" + "="*60)
print(" CRÉATION DES DATASETS")
print("="*60)

# ============================================
# 1. VERSION AVEC TOUTES LES FEATURES (85)
# ============================================

print("\n 1. Version avec toutes les features (85) :")
print(f"   X_train_full : {X_train.shape}")
print(f"   X_test_full : {X_test.shape}")

# Sauvegarder la version complète
X_train_full = X_train.copy()
X_test_full = X_test.copy()

# ============================================
# 2. VERSION AVEC FEATURES SÉLECTIONNÉES (40)
# ============================================

print("\n 2. Version avec features sélectionnées (40) :")
print(f"   X_train_selected : {X_train_selected.shape}")
print(f"   X_test_selected : {X_test_selected.shape}")

# ============================================
# 3. SAUVEGARDE DES DEUX VERSIONS
# ============================================

import os
os.makedirs('../data/processed', exist_ok=True)

# 3.1 Version complète (85 features)
X_train_full.to_csv('../data/processed/X_train_full.csv', index=False)
X_test_full.to_csv('../data/processed/X_test_full.csv', index=False)
print("\n Version complète (85 features) sauvegardée :")
print("   - data/processed/X_train_full.csv")
print("   - data/processed/X_test_full.csv")

# 3.2 Version sélectionnée (40 features)
X_train_selected.to_csv('../data/processed/X_train_selected.csv', index=False)
X_test_selected.to_csv('../data/processed/X_test_selected.csv', index=False)
print("\n Version sélectionnée (40 features) sauvegardée :")
print("   - data/processed/X_train_selected.csv")
print("   - data/processed/X_test_selected.csv")

# 3.3 Sauvegarder y_train et y_test (communs)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
print("\n y_train et y_test sauvegardés :")
print("   - data/processed/y_train.csv")
print("   - data/processed/y_test.csv")

# 3.4 Sauvegarder la liste des features sélectionnées
with open('../data/processed/selected_features.txt', 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")
print("\n Liste des features sélectionnées sauvegardée :")
print("   - data/processed/selected_features.txt")